# H9FR2 Financial Regulations CA1 — Python Verification Notebook
## National College of Ireland | MSc FinTech (MSCFTD1)
### Student: Hope Eneojo Ocholi | Student ID: 24338761
### Lecturer: Dr Brian Byrne | Submission Date: 31 March 2026

---

**Purpose:** This notebook provides independent Python verification of all Excel/VBA computations in the master workbook. Each section mirrors the corresponding Excel sheet with identical parameter values, enabling cross-validation to confirm numerical accuracy. All functions are written from first principles to demonstrate understanding of underlying financial mathematics.

**Structure:**
- Q5(i) Present Value of an Annuity
- Q5(ii) Bond Pricing, Duration, and Sensitivity Analysis (incl. Hopewell & Kaufman 1973 replication)
- Q5(iii) Yield to Maturity via Bisection Method
- Q5(iv) Mortgage Repayment and Amortization
- Q5(v) Term Structure Estimation (Continuous Rates)
- Q6(i) Black (1976) Option on Bond Futures
- Q6(ii) Portfolio Covariance, VaR, and CVaR (with R verification)
- Q6(iii) Executive Stock Options — Hull-White (2004) Fair Value and Sensitivity


In [ ]:
# Core imports — all computations from first principles
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import log, sqrt, exp, erf
from scipy.stats import norm

pd.set_option('display.float_format', '{:.6f}'.format)
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 11})
print("Environment ready — all libraries loaded.")


---
## Q5(i) Present Value of an Annuity

For an ordinary annuity with level payment $C$, periodic discount rate $r$, and $n$ periods:

$$PV_{\text{ordinary}} = C \times \frac{1 - (1+r)^{-n}}{r}$$

For an annuity due (payments at start of period):

$$PV_{\text{due}} = PV_{\text{ordinary}} \times (1+r)$$

**Parameters (matching Excel sheet Q5_i_Annuity):**
- Payment per period: €1,250
- Annual discount rate: 6.4%
- Years: 12
- Payments per year: 4 (quarterly)


In [ ]:
# Q5(i) — Annuity Present Value Function
def pv_annuity(payment, annual_rate, years, freq, due=False):
    """
    Calculate present value of an annuity.

    Parameters:
        payment: periodic payment amount
        annual_rate: annual discount rate
        years: number of years
        freq: payments per year
        due: True for annuity due, False for ordinary
    Returns:
        Present value
    """
    r = annual_rate / freq
    n = years * freq
    if r == 0:
        pv_ord = payment * n
    else:
        pv_ord = payment * (1 - (1 + r) ** (-n)) / r
    return pv_ord * (1 + r) if due else pv_ord

# Parameters matching Excel
PMT, RATE, YRS, FREQ = 1250, 0.064, 12, 4

pv_ord = pv_annuity(PMT, RATE, YRS, FREQ, due=False)
pv_due = pv_annuity(PMT, RATE, YRS, FREQ, due=True)

print(f"Periodic rate:          {RATE/FREQ:.6f}")
print(f"Total periods:          {YRS*FREQ}")
print(f"PV (ordinary annuity):  €{pv_ord:,.2f}")
print(f"PV (annuity due):       €{pv_due:,.2f}")
print(f"Due premium (1+r):      {pv_due/pv_ord:.6f}  [expected: {1+RATE/FREQ:.6f}]")


In [ ]:
# Rate sensitivity analysis — matches Excel D7:E15
rates = np.array([0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.10, 0.11])
pv_by_rate = [pv_annuity(PMT, r, YRS, FREQ) for r in rates]

rate_df = pd.DataFrame({'Annual Rate': [f'{r:.0%}' for r in rates], 'PV Ordinary': pv_by_rate})
print("Rate Sensitivity (matching Excel columns D–E):")
print(rate_df.to_string(index=False))
print(f"\nConvexity visible: PV drops {pv_by_rate[0]-pv_by_rate[1]:.2f} from 3%→4% vs {pv_by_rate[-2]-pv_by_rate[-1]:.2f} from 10%→11%")


In [ ]:
# Tenor sensitivity — matches Excel G7:H16
tenors = np.array([2, 4, 6, 8, 10, 12, 14, 16, 18, 20])
pv_by_tenor = [pv_annuity(PMT, RATE, t, FREQ) for t in tenors]

tenor_df = pd.DataFrame({'Years': tenors, 'PV Ordinary': pv_by_tenor})
print("Tenor Sensitivity (matching Excel columns G–H):")
print(tenor_df.to_string(index=False))


In [ ]:
# Visualisation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(rates*100, pv_by_rate, 'o-', color='#1F3864', linewidth=2)
ax1.set_title('Q5(i): PV Sensitivity to Discount Rate', fontweight='bold')
ax1.set_xlabel('Annual Discount Rate (%)')
ax1.set_ylabel('Present Value (€)')
ax1.grid(True, alpha=0.3)
ax1.ticklabel_format(style='plain', axis='y')

ax2.plot(tenors, pv_by_tenor, 's-', color='#2E75B6', linewidth=2)
ax2.set_title('Q5(i): PV Sensitivity to Tenor', fontweight='bold')
ax2.set_xlabel('Years')
ax2.set_ylabel('Present Value (€)')
ax2.grid(True, alpha=0.3)
ax2.ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()



**Interpretation.** The relationship is inverse and convex: as the discount rate rises, the present value falls, but not linearly.  
That is precisely why fixed-income style liabilities become more valuable when rates decline.

The sensitivity analysis shows that the present value of the annuity is inversely correlated to the discount rate. Higher annual discount rates lead to lower present values as future cash flows are discounted more significantly. This is consistent with the concept of the time value of money, in which an increase in the required rate of return leads to a lower present value of future cash flows. The continuous downward sloping curve indicates that annuity pricing is very sensitive to interest rates, especially when it comes to financial planning, valuing bonds and pricing loans.

---
## Q5(ii) Bond Pricing, Duration, and Sensitivity Analysis

Bond price as the sum of discounted cash flows:

$$P = \sum_{t=1}^{mT} \frac{C/m}{(1+y/m)^t} + \frac{F}{(1+y/m)^{mT}}$$

Macaulay duration:

$$D_{\text{Mac}} = \frac{1}{P} \sum_{t=1}^{mT} \frac{t}{m} \cdot \frac{CF_t}{(1+y/m)^t}$$

**Parameters (matching Excel Q5_ii_Bond_Duration):**
- Face: €1,000 | Coupon: 5.8% | YTM: 6.2% | Maturity: 7 years | Semi-annual


For rate-risk management, regulators care not only about price but also duration, because duration approximates how strongly prices react to yield changes.  
This is central to the asset-liability mismatch issues highlighted by the SVB episode.

In [ ]:
# Q5(ii) — Bond pricing and duration from first principles
def bond_price(face, coupon_rate, ytm, years, freq=2):
    n = int(round(years * freq))
    c = face * coupon_rate / freq
    r = ytm / freq
    price = sum(c / (1 + r)**t for t in range(1, n+1))
    price += face / (1 + r)**n
    return price

def macaulay_duration(face, coupon_rate, ytm, years, freq=2):
    n = int(round(years * freq))
    c = face * coupon_rate / freq
    r = ytm / freq
    px = bond_price(face, coupon_rate, ytm, years, freq)
    weighted = sum((t/freq) * c / (1+r)**t for t in range(1, n+1))
    weighted += years * face / (1+r)**n
    return weighted / px

def modified_duration(face, coupon_rate, ytm, years, freq=2):
    mac = macaulay_duration(face, coupon_rate, ytm, years, freq)
    return mac / (1 + ytm/freq)

# Parameters matching Excel
FACE, CPN, YTM_B, MAT, FREQ_B = 1000, 0.058, 0.062, 7, 2

px = bond_price(FACE, CPN, YTM_B, MAT, FREQ_B)
mac_d = macaulay_duration(FACE, CPN, YTM_B, MAT, FREQ_B)
mod_d = modified_duration(FACE, CPN, YTM_B, MAT, FREQ_B)
dv01 = mod_d * px / 10000

print(f"Bond price:            €{px:,.4f}")
print(f"Macaulay duration:     {mac_d:.4f} years")
print(f"Modified duration:     {mod_d:.4f} years")
print(f"DV01:                  €{dv01:.4f}")
print(f"Coupon/period:         €{FACE*CPN/FREQ_B:.2f}")


In [ ]:
# Full cash flow schedule
schedule = []
n = int(MAT * FREQ_B)
r = YTM_B / FREQ_B
c = FACE * CPN / FREQ_B

for t in range(1, n+1):
    time_yrs = t / FREQ_B
    coupon = c
    principal = FACE if t == n else 0
    total_cf = coupon + principal
    df = (1 + r) ** (-t)
    pv_cf = total_cf * df
    time_pv = time_yrs * pv_cf
    schedule.append([t, time_yrs, coupon, principal, total_cf, df, pv_cf, time_pv])

sched_df = pd.DataFrame(schedule, columns=['Period', 'Time(yrs)', 'Coupon', 'Principal', 'TotalCF', 'DF', 'PV_CF', 'Time*PV'])
print("Cash Flow Schedule:")
print(sched_df.to_string(index=False, float_format='%.4f'))
print(f"\nSum PV_CF = {sched_df['PV_CF'].sum():.4f}  [should match bond price: {px:.4f}]")
print(f"Mac. Duration = Sum(Time*PV)/Price = {sched_df['Time*PV'].sum():.4f}/{px:.4f} = {sched_df['Time*PV'].sum()/px:.4f}")


In [ ]:
# Yield sensitivity
yields = np.array([0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09])
yield_data = []
for y in yields:
    p = bond_price(FACE, CPN, y, MAT, FREQ_B)
    md_val = modified_duration(FACE, CPN, y, MAT, FREQ_B)
    yield_data.append([y, p, md_val])

yield_df = pd.DataFrame(yield_data, columns=['YTM', 'Price', 'Mod Duration'])
print("Yield Sensitivity:")
print(yield_df.to_string(index=False, float_format='%.4f'))


In [ ]:
# Hopewell & Kaufman (1973) duration replication table
# Duration for various coupon/maturity combinations at YTM = 6.2%
coupons_hk = [0.01, 0.04, 0.058, 0.08, 0.12, 0.20]
mats_hk = [1, 5, 10, 15, 20, 30]

hk_data = {}
for cpn in coupons_hk:
    row = []
    for mat in mats_hk:
        d = macaulay_duration(FACE, cpn, YTM_B, mat, FREQ_B)
        row.append(d)
    hk_data[f'{cpn:.1%}'] = row

hk_df = pd.DataFrame(hk_data, index=[f'{m}yr' for m in mats_hk]).T
print("Hopewell & Kaufman (1973) Duration Table (Macaulay, years):")
print("YTM = 6.2%, Semi-annual coupons")
print(hk_df.to_string(float_format='%.2f'))
print("\nKey insight: Duration is driven by coupon and maturity jointly.")
print("Lower coupons → longer duration (more weight on final principal).")
print("This table replicates the logic from the original H&K paper.")


In [ ]:
# Price-yield and duration charts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(yield_df['YTM']*100, yield_df['Price'], 'o-', color='#1F3864', linewidth=2)
ax1.set_title('Q5(ii): Bond Price–Yield Relationship', fontweight='bold')
ax1.set_xlabel('Yield to Maturity (%)')
ax1.set_ylabel('Bond Price (€)')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=FACE, color='grey', linestyle='--', alpha=0.5, label='Par')
ax1.legend()

ax2.plot(yield_df['YTM']*100, yield_df['Mod Duration'], 's-', color='#2E75B6', linewidth=2)
ax2.set_title('Q5(ii): Modified Duration vs Yield', fontweight='bold')
ax2.set_xlabel('Yield to Maturity (%)')
ax2.set_ylabel('Modified Duration (years)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


**Relevance of interpretation and relevance of SVB.**

Duration involves calculation of weighted-average duration to collect cash flows and estimates the rate of percentage price change with a unit change in yield. According to the Hopewell-Kaufman (1973) table, the sensitivity of prices depends on the duration and not just the maturity since the level of coupon determines the weighting of the cash-flows. Silicon Valley Bank (March 2023) had ~\$91bn of HTM securities with a much longer maturity than the length of its deposit-liability, which led to an unrealized loss of ~\$17bn and, on the insolvency of the bank as a result of deposit outflows, to asset sales. This ascertains Basel III IRRBB duration-gap stress testing of rate shocks of +-200bps.

**An analysis**

- Longer maturity typically increases duration, as greater value is received later in the future.  
- Higher coupons shorten duration since more money is received sooner.
- Lower yields tend to increase duration, as distant cash flows are discounted less heavily.  
- Regulators are concerned about duration matching since a bank backed by flight-prone deposits but holding long-duration assets may experience a severe mark-to-market capital shock when interest rates increase.


The accurate bond price vs yield graph slopes downward. This represents the inverse relationship between bond prices and yields: as the bond's yield to maturity rises, the present value of its future coupon and principal payments falls. As a result, the chart confirms standard fixed-income valuation theory while also supporting duration-based sensitivity analysis.

**Hopewell and Kaufman (1973) logic**

The main perspective from this analysis is that price sensitivity is driven by duration rather than by maturity alone.  
The table above proves exactly this relationship:  
For two bonds with different coupons, the bond with the lower coupon may have the greater duration even at the same maturity.

The results are in line with Hopewell and Kaufman (1973) model of bond price and term, as calculated by the present-value weighting of cash flows. Increasing yields reduce the influence of cash flows further into the future, which causes bond prices and duration to decline. This accounts for the inverse price-yield relationship and the reduction in duration.


This report demonstrates that duration is not just a mathematical construct, but also a valuable risk-mitigation measure. Longer-term bonds are more sensitive to changes in interest rates and can lead to mark-to-market losses for financial institutions. As was apparent in the Silicon Valley Bank (SVB) case, long-term assets and short-term liabilities were a significant source of risk when interest rates spiked. Policy makers emphasise duration matching to minimise interest-rate risk and preserve financial stability.

The duration calculations are based on the present-value weighting model described by Hopewell and Kaufman (1973) in which the contribution of each cash flow to duration is weighted by the present value of the cash flow and the time to receipt. This helps explain why long-term and low-coupon bonds have a longer duration and interest-rate risk.

---
## Q5(iii) Yield to Maturity via Bisection Method

The bisection method finds the YTM (root of P(y) − Market Price = 0) by iteratively halving a bracketing interval.

**Parameters (matching Excel Q5_iii_YTM):**
- Face: €1,000 , Coupon: 4.75% , Market price: €978.40 , Years: 6, Semi-annual
- Initial bracket: [1%, 15%]


In [ ]:
# Q5(iii) — Bisection method for YTM
def bisection_ytm(face, coupon_rate, market_price, years, freq=2,
                   low=0.01, high=0.15, tol=1e-8, max_iter=30):
    iterations = []
    for i in range(1, max_iter+1):
        mid = (low + high) / 2
        price_mid = bond_price(face, coupon_rate, mid, years, freq)
        error = price_mid - market_price
        decision = "Converged" if abs(error) < tol else ("Low=Mid" if error > 0 else "High=Mid")
        iterations.append([i, low, high, mid, price_mid, error, decision])
        if abs(error) < tol:
            break
        if error > 0:
            low = mid
        else:
            high = mid
    return mid, iterations

# Parameters matching Excel
FACE_Y, CPN_Y, MKT_PX, YRS_Y, FREQ_Y = 1000, 0.0475, 978.40, 6, 2

ytm_conv, iters = bisection_ytm(FACE_Y, CPN_Y, MKT_PX, YRS_Y, FREQ_Y)

iter_df = pd.DataFrame(iters, columns=['Iter', 'Low', 'High', 'Mid', 'Price(mid)', 'Error', 'Decision'])
print("Bisection Convergence Table (matching Excel A17:G30):")
print(iter_df.to_string(index=False, float_format='%.8f'))
print(f"\nConverged YTM: {ytm_conv:.8f} ({ytm_conv*100:.4f}%)")
print(f"Verification: Price at converged YTM = €{bond_price(FACE_Y, CPN_Y, ytm_conv, YRS_Y, FREQ_Y):.4f}")
print(f"Error: {bond_price(FACE_Y, CPN_Y, ytm_conv, YRS_Y, FREQ_Y) - MKT_PX:.8f}")


In [ ]:
# Convergence plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(iter_df['Iter'], iter_df['Error'].abs(), 'o-', color='#C00000', linewidth=2)
ax.set_title('Q5(iii): Bisection Convergence — |Error| vs Iteration', fontweight='bold')
ax.set_xlabel('Iteration')
ax.set_ylabel('|Pricing Error| (€)')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.axhline(y=1e-8, color='green', linestyle='--', alpha=0.7, label='Tolerance (10⁻⁸)')
ax.legend()
plt.tight_layout()
plt.show()


**Interpretation.**
Bisection methodology converges since the yields of bonds are decreasing continuously. Log-linear convergence is half the error per iteration and machine tolerant at approximately 27 iterations. Although the convergence of Newton-Raphson may be quicker, the use of bisection is better in pedagogical settings since convergence and transparency are ensured.

The Python model estimates yield to maturity using the method of bisection in Excel. The monotonic nature of bond prices as yields increase permits the root to be found within a range between low and high guesses. The bond is repriced at each iteration to check the yield in the middle of the range. If the resulting price exceeds the market price, then the yield guess is too low and the lower bound is increased, otherwise the upper bound is decreased. For the inputs shown in Excel, the model finds the YTM to be about 5.17% (a discount bond). The repricing calculator suggests that the YTM guess is close to the market price.

---
## Q5(iv) Mortgage Repayment and Amortization

Monthly payment formula:

$$M = P \times \frac{r(1+r)^n}{(1+r)^n - 1}$$

**Parameters (matching Excel Q5_iv_Mortgage):**
- Principal: €335,000 | Annual rate: 4.65% | Years: 25 | Monthly | Extra prepayment: €150/month


In [ ]:
# Q5(iv) — Mortgage payment and amortization
def mortgage_payment(principal, annual_rate, years, freq=12):
    r = annual_rate / freq
    n = years * freq
    return principal * r * (1+r)**n / ((1+r)**n - 1)

PRINC, RATE_M, YRS_M, FREQ_M, EXTRA = 335000, 0.0465, 25, 12, 150

pmt = mortgage_payment(PRINC, RATE_M, YRS_M, FREQ_M)
print(f"Monthly rate:       {RATE_M/FREQ_M:.6f}")
print(f"Scheduled payment:  €{pmt:,.2f}")
print(f"Payment incl. prepay: €{pmt+EXTRA:,.2f}")


In [ ]:
# Full amortization schedule
schedule_m = []
balance = PRINC
for t in range(1, YRS_M*FREQ_M + 1):
    if balance <= 0:
        break
    interest = balance * RATE_M / FREQ_M
    sched_princ = pmt - interest
    extra = EXTRA if balance > 0 else 0
    total_princ = min(balance, sched_princ + extra)
    closing = max(0, balance - total_princ)
    schedule_m.append([t, balance, pmt, interest, sched_princ, extra, total_princ, closing])
    balance = closing

mort_df = pd.DataFrame(schedule_m, columns=['Pmt#', 'Opening', 'SchedPmt', 'Interest', 'SchedPrinc', 'ExtraPrepay', 'TotalPrinc', 'Closing'])

print(f"First 12 payments:")
print(mort_df.head(12).to_string(index=False, float_format='%.2f'))
print(f"\nLast payment at month: {mort_df.iloc[-1]['Pmt#']:.0f}")
print(f"Total interest paid:   €{mort_df['Interest'].sum():,.2f}")
print(f"Total principal paid:  €{mort_df['TotalPrinc'].sum():,.2f}")
print(f"Months saved by €{EXTRA}/mo prepayment: {YRS_M*FREQ_M - len(mort_df)}")


In [ ]:
# Rate sensitivity
rate_vals = np.array([0.03, 0.04, 0.0465, 0.05, 0.06, 0.07])
rate_sens = pd.DataFrame({
    'Rate': [f'{r:.2%}' for r in rate_vals],
    'Monthly Pmt': [mortgage_payment(PRINC, r, YRS_M) for r in rate_vals],
    'Total Interest': [mortgage_payment(PRINC, r, YRS_M)*YRS_M*FREQ_M - PRINC for r in rate_vals]
})
print("Rate Sensitivity:")
print(rate_sens.to_string(index=False, float_format='%.2f'))


In [ ]:
# Amortization visualisation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(mort_df['Pmt#'], mort_df['Closing'], color='#1F3864', linewidth=2)
ax1.set_title('Q5(iv): Mortgage Balance Over Time', fontweight='bold')
ax1.set_xlabel('Payment Number')
ax1.set_ylabel('Closing Balance (€)')
ax1.grid(True, alpha=0.3)
ax1.ticklabel_format(style='plain', axis='y')

plot_df = mort_df[mort_df['Closing'] > 0]
ax2.fill_between(plot_df['Pmt#'], plot_df['Interest'], alpha=0.5, color='#C00000', label='Interest')
ax2.fill_between(plot_df['Pmt#'], plot_df['TotalPrinc'], alpha=0.5, color='#2E75B6', label='Principal')
ax2.set_title('Q5(iv): Interest vs Principal Decomposition', fontweight='bold')
ax2.set_xlabel('Payment Number')
ax2.set_ylabel('Amount (€)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


**IO/PO strip note.** Interest-only strips derive their value from the interest component; they have negative duration since prepayment acceleration (when rates fall) reduces the interest stream. principle-only strips benefit from speedier prepayment because the principle is returned sooner and discounted less steeply. In practice, IO/PO strips are traded as FNMA/FHLMC trust classes and used by mortgage servicers and ALM desks to hedge prepayment and interest-rate risks.

The amortization schedule illustrates the typical mechanics of a mortgage payment. Since the outstanding debt is high at beginning of the life of loan, the interest component of the scheduled payment is high. The interest component decreases and the principal component rises each term. The additional monthly prepayment of 150 Euros reduces the effective lifespan of the loan and accelerates the amortization by increasing the total principle repaid per term.
The interest rate sensitivity analysis shows that higher interest rates on the loan increase the scheduled payment and slow down the rate at which the principal is repaid in the early term of the loan, due to the higher interest component of the scheduled payment. This has implications for structured mortgages as it impacts cash flows. So, faster repayment and prepayment can reduce the value of IO strips by reducing the interest cash flows, but increase the value of PO strips by returning principal sooner. Indeed, IO and PO strips are priced by traders under a range of assumptions about rates and borrower prepayments because these embedded optionality effects can dramatically increase or decrease the value and risk of the strips.


```

```


In summary, the Python program accurately validates the Excel mortgage repayment and amortization model from Q5(iv). For the same mortgage inputs, the calculated repayment and amortization tables are the same for Excel, VBA and Python. Our analysis also highlights the impact of prepayment and interest rates on mortgage valuation. This is particularly relevant to IO and PO strips, which are sensitive to interest and principal cash flow timing. This task therefore combines precise calculations with financial insights, which is in line with the estimating nature of the module.

---
## Q5(v) Term Structure Estimation (Continuous Rates)

Continuously compounded spot rate from discount factor:

$$r(t) = -\frac{\ln(DF_t)}{t}$$

**Parameters (matching Excel Q5_v_TermStructure):**


In [ ]:
# Q5(v) — Term structure estimation
maturity = np.array([0.5, 1, 2, 3, 5, 7, 10])
disc_factors = np.array([0.981, 0.9608, 0.9222, 0.8869, 0.8167, 0.7558, 0.6554])

spot_rates = -np.log(disc_factors) / maturity

# Linear regression: r(t) = a + b*t
slope, intercept = np.polyfit(maturity, spot_rates, 1)
fitted = intercept + slope * maturity

# Forward rates
forward_rates = []
for i in range(len(maturity)-1):
    f = (spot_rates[i+1]*maturity[i+1] - spot_rates[i]*maturity[i]) / (maturity[i+1] - maturity[i])
    forward_rates.append(f)

term_df = pd.DataFrame({
    'Maturity': maturity,
    'DF': disc_factors,
    'Spot Rate (cc)': spot_rates,
    'Fitted Rate': fitted
})

print("Term Structure (matching Excel Q5_v_TermStructure):")
print(term_df.to_string(index=False, float_format='%.6f'))
print(f"\nRegression slope:     {slope:.6f}")
print(f"Regression intercept: {intercept:.6f}")
print(f"R-squared:            {np.corrcoef(maturity, spot_rates)[0,1]**2:.6f}")
print(f"\nForward Rates:")
for i, f in enumerate(forward_rates):
    print(f"  {maturity[i]:.1f}y → {maturity[i+1]:.1f}y:  {f:.4%}")


In [ ]:
# Term structure chart
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(maturity, spot_rates*100, 'o-', color='#1F3864', linewidth=2, label='Observed spot rate')
ax.plot(maturity, fitted*100, '--', color='#C00000', linewidth=2, label='Fitted (linear regression)')
ax.set_title('Q5(v): Term Structure — Observed vs Fitted', fontweight='bold')
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Spot Rate (%, continuous)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**Interpretation.**
To estimate the term structure of interest rates, discount factors were used to calculate continuously compounded spot rates r(t)=−ln(DFt)/t. We then estimated a linear regression of the spot rates on maturity to obtain a fitted line that resembles a small version of a JP Morgan mapping. The regression resulted in a positive slope and an intercept of a little under 4%, implying a slightly upward-sloping yield curve. This means that longer-dated cash flows must be compensated with slightly higher yields than shorter-dated cash flows, suggesting a positive term premium.

The method is useful as it translates discount factors into an intuitive and interpretable yield curve that can be compared across different maturities. It's straightforward and consistent to implement in Excel, VBA and Python, and reproducible. But the method is imperfect: a linear regression will not reveal more complicated curve shapes such as humps, kinks or inversions. However, the regression line is only a first approximation: more sophisticated methods such as splines or Nelson-Siegel models are used in practice. In summary, the results reveal a stable curve with a level of 4% and a modest upward slant, which is economically sensible given the modest compensation for lending for a longer period.

---
## Q6(i) Black (1976) for a Call Option on a Bond Futures Contract

**Objective.**

This analysis applies the Black (1976) model to value a European call option on a bond futures contract. The model is applicable since the underlying instrument is a futures price, rather than a spot bond price. The option value is then assessed for sensitivity to futures price and volatility, and the results are interpreted in terms of interest-rate risk management.

$$C = e^{-rT}[F \cdot N(d_1) - K \cdot N(d_2)]$$

where $d_1 = \frac{\ln(F/K) + \frac{1}{2}\sigma^2 T}{\sigma\sqrt{T}}$, $d_2 = d_1 - \sigma\sqrt{T}$

**Parameters (matching Excel Q6_i_Black76):**
- F=108.20, K=107.00, r=3.25%, T=0.75yr, σ=17%


In [ ]:
# Q6(i) — Black (1976) model
def black76_call(F, K, r, T, sigma):
    d1 = (log(F/K) + 0.5*sigma**2*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return exp(-r*T) * (F*norm.cdf(d1) - K*norm.cdf(d2))

def black76_put(F, K, r, T, sigma):
    d1 = (log(F/K) + 0.5*sigma**2*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return exp(-r*T) * (K*norm.cdf(-d2) - F*norm.cdf(-d1))

F, K, r_b, T_b, sigma_b = 108.20, 107.00, 0.0325, 0.75, 0.17

d1 = (log(F/K) + 0.5*sigma_b**2*T_b) / (sigma_b*sqrt(T_b))
d2 = d1 - sigma_b*sqrt(T_b)
call_val = black76_call(F, K, r_b, T_b, sigma_b)
put_val = black76_put(F, K, r_b, T_b, sigma_b)

print(f"d1:                    {d1:.6f}")
print(f"d2:                    {d2:.6f}")
print(f"N(d1):                 {norm.cdf(d1):.6f}")
print(f"N(d2):                 {norm.cdf(d2):.6f}")
print(f"Call value:            €{call_val:.4f}")
print(f"Put value:             €{put_val:.4f}")
print(f"\nPut-call parity check:")
print(f"  C - P              = {call_val - put_val:.4f}")
print(f"  e^(-rT)(F-K)       = {exp(-r_b*T_b)*(F-K):.4f}")
print(f"  Difference:          {abs(call_val-put_val - exp(-r_b*T_b)*(F-K)):.10f}")


In [ ]:
# Sensitivity tables — matching Excel
futures_vals = np.array([100, 102, 104, 106, 108, 110, 112, 114], dtype=float)
vol_vals = np.array([0.10, 0.12, 0.14, 0.16, 0.18, 0.20, 0.22, 0.24, 0.26, 0.28, 0.30])

futures_sens = pd.DataFrame({
    'Futures Price': futures_vals,
    'Call Value': [black76_call(f, K, r_b, T_b, sigma_b) for f in futures_vals]
})
vol_sens = pd.DataFrame({
    'Volatility': vol_vals,
    'Call Value': [black76_call(F, K, r_b, T_b, v) for v in vol_vals]
})

print("Futures Price Sensitivity:")
print(futures_sens.to_string(index=False, float_format='%.4f'))
print("\nVolatility Sensitivity:")
print(vol_sens.to_string(index=False, float_format='%.4f'))


In [ ]:
# Charts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(futures_sens['Futures Price'], futures_sens['Call Value'], 'o-', color='#1F3864', linewidth=2)
ax1.set_title('Q6(i): Black-76 Call Value vs Futures Price', fontweight='bold')
ax1.set_xlabel('Futures Price')
ax1.set_ylabel('Call Value (€)')
ax1.axvline(x=K, color='grey', linestyle='--', alpha=0.5, label=f'Strike = {K}')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(vol_sens['Volatility']*100, vol_sens['Call Value'], 'o-', color='#2E75B6', linewidth=2)
ax2.set_title('Q6(i): Black-76 Call Value vs Volatility', fontweight='bold')
ax2.set_xlabel('Volatility (%)')
ax2.set_ylabel('Call Value (€)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


**Interpretation.** The call value grows convexly with the futures price (delta approaches one deep ITM) and linearly with volatility (vega). Bond futures options protect against interest rate risk by gaining value when futures rise (rates decrease), offsetting losses on short cash-bond positions. To verify early exercise at each node, American parallels require binomial tree valuation.

An option on a bond futures contract was valued using the Black (1976) model with futures price, strike, risk-free rate and volatility of 108.20, 107.00, 3.25% and 17% respectively, and maturity 0.75 years away. The model produced
𝑑1≈0.15, 𝑑2≈0.00, and a call value of about €6.77, with the corresponding put value of about €5.59. These values are consistent with the option being slightly in the money and with there being considerable uncertainty at expiry.

We can perform sensitivity analysis to check that the call value rises with an increase in the futures price and in volatility. This is consistent with what is expected of option values: a higher futures price increases the expected intrinsic value, and higher volatility increases the chance of a price change. The Black model is suitable because the underlying asset is a futures contract, rather than the spot bond. It is also relatively simple to use for pricing and for application in Excel, VBA and Python.

The American version cannot be priced in closed form, in general, because it involves early exercise. The latter two methods are employed to compare the value of holding the futures option versus exercising it early. From a risk-management perspective, bond futures options can hedge interest-rate risk because they offer partial protection against adverse futures price changes associated with bond yields. But although Black (1976) is a useful benchmark model, it ignores changing volatility, yield-curve shape factors and bond-futures delivery options, so it should be viewed as an approximate market model.

In [ ]:
#!pip install yfinance
#https://pypi.org/project/yfinance/

In [ ]:
import pandas as pd
import yfinance as yf

tickers_list = ['NVDA', 'NFLX', 'AMZN', 'SNAP', 'RIG', 'CCL' , 'NIO' , 'NOK','MARA','U']

# Fetch the data with auto_adjust set to False
data = yf.download(tickers_list, start='2023-05-26', end='2024-05-27', auto_adjust=False)

# Access the 'Adj Close' column
adj_close_data = data['Adj Close']

# Print the last 10 rows
print(adj_close_data.tail(10))


In [ ]:
adj_close_data.to_excel("stock_data.xlsx")
print("File saved successfully!")

In [ ]:
import os
print(os.getcwd())

In [ ]:
!ls

In [ ]:
import os
os.listdir()

In [ ]:
#Load the file
df = pd.read_excel("stock_data.xlsx")
df.head()

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

stock_cols = ["AMZN", "CCL", "MARA", "NFLX", "NIO", "NOK", "NVDA", "RIG", "SNAP", "U"]

for col in stock_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
        .astype(float) / 100
    )

df = df.sort_values("Date").reset_index(drop=True)

print(df.head())

In [ ]:
df.to_excel("cleaned_portfolio_data.xlsx", index=False)

In [ ]:
#Plot the portfolio line chart
plt.figure(figsize=(10, 6))
for col in stock_cols:
    plt.plot(df["Date"], df[col], label=col)

plt.title("Q6(ii): Portfolio Price Series")
plt.xlabel("Date")
plt.ylabel("Price")
plt.grid(True)
plt.legend(ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
#Compute the daily returns
returns = df[stock_cols].pct_change().dropna()

print("Daily Returns")
print(returns.head())

---
## Q6(ii) Portfolio Risk: Covariance, VaR, and CVaR

**Portfolio:** 10 equally-weighted stocks (NVDA, NFLX, AMZN, SNAP, RIG, CCL, NIO, NOK, MARA, U)
**Data:** Daily prices from Q6_ii_RawData sheet (replace with live yfinance data for submission)
**Portfolio value:** €2,500,000 | Confidence: 99% | Horizon: 10 days

$$\text{VaR}_{10d,1\%} = z_{0.99} \cdot \sigma_p \cdot \sqrt{10} \cdot V$$


In [ ]:
# Q6(ii) — Portfolio VaR and CVaR from REAL stock data
# The Excel Q6_ii_RawData sheet contains daily LOG RETURNS computed from
# adjusted close prices downloaded via yfinance for 10 stocks over 1 year.
#
# In your Colab submission, run:
#   import yfinance as yf
#   tickers = ['AMZN','CCL','MARA','NFLX','NIO','NOK','NVDA','RIG','SNAP','U']
#   data = yf.download(tickers, start='2023-05-25', end='2024-05-25', auto_adjust=True)
#   prices = data['Close'][tickers].dropna()
#   log_returns = np.log(prices).diff().dropna()
#   log_returns.to_excel("/content/cleaned_portfolio_data.xlsx")  # Export for R verification

tickers = ['AMZN','CCL','MARA','NFLX','NIO','NOK','NVDA','RIG','SNAP','U']

# Read from Excel export (these are the same log returns in Q6_ii_RawData)
try:
    returns_df = pd.read_excel("daily_log_returns.xlsx")[tickers]
except:
    # Alternative: compute from raw prices
    prices_raw = pd.read_excel("/content/cleaned_portfolio_data.xlsx")
    if 'Date' in prices_raw.columns:
        prices_raw = prices_raw.set_index('Date')
    returns_df = np.log(prices_raw[tickers]).diff().dropna()

print(f"Daily log returns: {returns_df.shape[0]} observations, {returns_df.shape[1]} stocks")
print(f"\nMean daily returns:")
print(returns_df.mean().to_string(float_format='%.6f'))
print(f"\nDaily volatilities:")
print(returns_df.std().to_string(float_format='%.6f'))


In [ ]:
# Variance-covariance matrix of daily log returns
# This matches Excel COVARIANCE.S formulas in Q6_ii_VaR_CVaR!F11:O20
# Excel computes: COVARIANCE.S(Q6_ii_Returns!B2:B252, Q6_ii_Returns!C2:C252)
# Python equivalent: returns_df.cov() (which uses sample covariance, N-1 denominator)
cov_matrix = returns_df.cov()

print("Variance-Covariance Matrix (daily log returns):")
print("Matches Excel Q6_ii_VaR_CVaR cells F11:O20")
print(cov_matrix.to_string(float_format='%.8f'))

print(f"\nDiagonal (variances):")
for t in tickers:
    print(f"  {t}: {cov_matrix.loc[t,t]:.8f}")


In [ ]:
# Portfolio return distribution with VaR/CVaR lines
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(port_returns, bins=40, density=True, alpha=0.7, color='#2E75B6', edgecolor='white')
var_line = np.percentile(port_returns, 1)
cvar_line = port_returns[port_returns <= var_line].mean()
ax.axvline(x=var_line, color='#C00000', linewidth=2, linestyle='--', label=f'VaR 1% = {abs(var_line)*100:.2f}%')
ax.axvline(x=cvar_line, color='#FF6600', linewidth=2, linestyle='-.', label=f'CVaR 1% = {abs(cvar_line)*100:.2f}%')
ax.set_title('Q6(ii): Daily Portfolio Return Distribution with VaR/CVaR', fontweight='bold')
ax.set_xlabel('Daily Log Return')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
#Portfolio weights
n_assets = len(stock_cols)
weights = np.array([1 / n_assets] * n_assets)

print("Portfolio Weights")
print(pd.Series(weights, index=stock_cols))

**R Verification (activate R magic in Colab)**

### R Verification (activate R magic in Colab)

```r
%%R
# Read stock prices exported from Python
StockPrices <- read.csv("StockPrices.csv")
# Compute log returns
returns_r <- diff(as.matrix(log(StockPrices[,-1])))
# Covariance matrix
cov_r <- cov(returns_r)
print(round(cov_r, 8))
# Equal weights
w <- rep(1/10, 10)
port_var_r <- t(w) %*% cov_r %*% w
cat("Portfolio daily variance (R):", port_var_r, "\n")
```

**Regulatory Perspective.** VaR at 99%/10-day is the Basel II/III Internal Models Approach market-risk capital charge. CVaR (Expected Shortfall) is preferred under the FRTB (Basel III.1) at 97.5% because it captures tail risk beyond VaR. The parametric approach assumes normality, which underestimates tail risk for fat-tailed distributions (see Hull RMFI Ch. 9).

In [ ]:
portfolio_value = 2_500_000
var_10d_eur = var_10d_1pct * portfolio_value

print(f"10-day VaR at 1% (€) = €{var_10d_eur:,.2f}")

Historical CVaR

CVaR at 1% is the average loss in the worst 1% of outcomes.

For 10-day scaling, a simple assignment-friendly approach is to scale daily returns by SQRT of 10.

In [ ]:
# Variance-covariance matrix
cov_matrix = returns_df.cov()
print("Variance-Covariance Matrix:")
print(cov_matrix.to_string(float_format='%.8f'))


In [ ]:
# Portfolio VaR and CVaR
n_assets = len(tickers)
weights = np.array([1/n_assets] * n_assets)
portfolio_value = 2_500_000

# Portfolio stats
port_var = weights @ cov_matrix.values @ weights
port_vol = np.sqrt(port_var)
port_returns = returns_df @ weights
port_mean = port_returns.mean()

# Parametric 10-day VaR at 99%
z_99 = norm.ppf(0.99)
var_10d_param = portfolio_value * (z_99 * port_vol * np.sqrt(10) - port_mean * 10)

# Parametric 10-day CVaR at 99%
phi_z = norm.pdf(z_99)
cvar_10d_param = portfolio_value * ((phi_z / 0.01) * port_vol * np.sqrt(10) - port_mean * 10)

# Historical 1-day VaR and CVaR
var_1d_hist = -np.percentile(port_returns, 1) * portfolio_value
threshold = np.percentile(port_returns, 1)
cvar_1d_hist = -port_returns[port_returns <= threshold].mean() * portfolio_value

print("Portfolio Risk Summary")
print("=" * 55)
print(f"Portfolio daily mean return:     {port_mean:.8f}")
print(f"Portfolio daily variance:        {port_var:.8f}")
print(f"Portfolio daily volatility:      {port_vol:.6f} ({port_vol*100:.2f}%)")
print(f"z(99%):                          {z_99:.4f}")
print(f"10-day parametric VaR at 1%:     €{var_10d_param:,.2f}")
print(f"10-day parametric CVaR at 1%:    €{cvar_10d_param:,.2f}")
print(f"1-day historical VaR at 1%:      €{var_1d_hist:,.2f}")
print(f"1-day historical CVaR at 1%:     €{cvar_1d_hist:,.2f}")

In [ ]:
# Portfolio return distribution with VaR/CVaR lines
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(port_ret_10d, bins=40, density=True, alpha=0.7, color='#2E75B6', edgecolor='white')
ax.axvline(x=-var_10d, color='#C00000', linewidth=2, linestyle='--', label=f'VaR 1% = {var_10d*100:.2f}%')
ax.axvline(x=cvar_1pct, color='#FF6600', linewidth=2, linestyle='-.', label=f'CVaR 1% = {abs(cvar_1pct)*100:.2f}%')
ax.set_title('Q6(ii): 10-Day Portfolio Return Distribution with VaR/CVaR', fontweight='bold')
ax.set_xlabel('10-Day Scaled Return')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### R Verification (activate R magic in Colab)

```r
%%R
# Read stock prices exported from Python
StockPrices <- read.csv("StockPrices.csv")
# Compute log returns
returns_r <- diff(as.matrix(log(StockPrices[,-1])))
# Covariance matrix
cov_r <- cov(returns_r)
print(round(cov_r, 8))
# Equal weights
w <- rep(1/10, 10)
port_var_r <- t(w) %*% cov_r %*% w
cat("Portfolio daily variance (R):", port_var_r, "\n")
```

**Regulatory Perspective.** VaR at 99%/10-day is the Basel II/III Internal Models Approach market-risk capital charge. CVaR (Expected Shortfall) is preferred under the FRTB (Basel III.1) at 97.5% because it captures tail risk beyond VaR. The parametric approach assumes normality, which underestimates tail risk for fat-tailed distributions (see Hull RMFI Ch. 9).

The analysis of portfolio risk was carried out by converting the daily stock prices to daily percentage returns and then calculating the variance-covariance matrix of the ten chosen stocks. The covariance method was then used to estimate portfolio volatility and a 10-day Value at Risk (VaR) at the 1% confidence level, assuming equal weights in the portfolio. We also estimated the Conditional Value at Risk (CVaR) to obtain the expected loss in the 1% worst case scenario.

VaR is helpful in that it gives a succinct estimate of the worst-case scenario loss over a particular time period and at a given level of confidence. But VaR does not provide information on losses above the threshold. CVaR uses the expected loss in the tail and is useful for tail-risk and stress testing. In this regard, CVaR is usually a more conservative risk measure.

---
## Q6(iii) Executive Stock Options — Hull-White (2004) Fair Value

The Hull-White (2004) ESO model modifies Black-Scholes-Merton by:
1. Replacing contractual life with expected exercise life
2. Applying a forfeiture probability during the vesting period
3. Including a continuous dividend yield

$$\text{ESO} = (1 - f)^{T_v} \times \text{BSM}(S, K, r, q, \sigma, T_e)$$

**Parameters (matching Excel Q6_iii_ESO):**


In [ ]:
# Q6(iii) — ESO Hull-White style fair value
def bsm_call_div(S, K, r, q, sigma, T):
    """Black-Scholes-Merton call with continuous dividend yield."""
    d1 = (log(S/K) + (r - q + 0.5*sigma**2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return S*exp(-q*T)*norm.cdf(d1) - K*exp(-r*T)*norm.cdf(d2)

def eso_fair_value(S, K, r, q, sigma, expected_life, vesting_years, forfeiture_rate):
    """
    Hull-White (2004) ESO fair value.
    Applies survival probability during vesting and uses expected exercise life.
    """
    survival = (1 - forfeiture_rate)**vesting_years
    bs_val = bsm_call_div(S, K, r, q, sigma, expected_life)
    eso_val = survival * bs_val
    return eso_val, bs_val, survival

# Parameters — read these from the Excel sheet
# Typical ESO parameters (matching Q6_iii_ESO input cells)
S = 42.0      # Current stock price
K = 40.0      # Strike
r_e = 0.031   # Risk-free rate
q_e = 0.012   # Dividend yield
sigma_e = 0.34 # Volatility
exp_life = 6.5 # Expected exercise life (vs 10yr contractual)
vest_yrs = 3.0 # Vesting period
forfeit = 0.035 # Annual forfeiture rate

eso, bs, surv = eso_fair_value(S, K, r_e, q_e, sigma_e, exp_life, vest_yrs, forfeit)

intrinsic_value = max(S - K, 0)

print("ESO Fair Value Calculation")
print("=" * 50)
print(f"Stock price:           €{S:.2f}")
print(f"Strike:                €{K:.2f}")
print(f"Risk-free rate:        {r_e:.2%}")
print(f"Dividend yield:        {q_e:.2%}")
print(f"Volatility:            {sigma_e:.2%}")
print(f"Expected life:         {exp_life:.1f} years")
print(f"Vesting period:        {vest_yrs:.1f} years")
print(f"Forfeiture rate:       {forfeit:.2%}")
print(f"\nBSM call value:        €{bs:.4f}")
print(f"Survival probability:  {surv:.6f}")
print(f"ESO fair value:        €{eso:.4f}")
print(f"Discount vs BSM:       {(1-eso/bs)*100:.2f}%")
print(f"Intrinsic value today  = €{intrinsic_value:,.2f}")


In [ ]:
# Sensitivity analysis
# Volatility
vol_vals = np.array([0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50])
vol_res = [(v, eso_fair_value(S, K, r_e, q_e, v, exp_life, vest_yrs, forfeit)[0]) for v in vol_vals]
vol_df = pd.DataFrame(vol_res, columns=['Volatility', 'ESO Value'])

# Expected life
life_vals = np.array([4.0, 5.0, 6.0, 6.5, 7.0, 8.0])
life_res = [(l, eso_fair_value(S, K, r_e, q_e, sigma_e, l, vest_yrs, forfeit)[0]) for l in life_vals]
life_df = pd.DataFrame(life_res, columns=['Expected Life', 'ESO Value'])

# Vesting period
vest_vals = np.array([1, 2, 3, 4, 5, 6], dtype=float)
vest_res = [(v, eso_fair_value(S, K, r_e, q_e, sigma_e, exp_life, v, forfeit)[0]) for v in vest_vals]
vest_df = pd.DataFrame(vest_res, columns=['Vesting Period', 'ESO Value'])

# Risk-free rate
rf_vals = np.array([0.01, 0.02, 0.031, 0.04, 0.05, 0.06])
rf_res = [(rf, eso_fair_value(S, K, rf, q_e, sigma_e, exp_life, vest_yrs, forfeit)[0]) for rf in rf_vals]
rf_df = pd.DataFrame(rf_res, columns=['Risk-Free Rate', 'ESO Value'])

print("Volatility Sensitivity:")
print(vol_df.to_string(index=False, float_format='%.4f'))
print("\nExpected Life Sensitivity:")
print(life_df.to_string(index=False, float_format='%.4f'))
print("\nVesting Period Sensitivity:")
print(vest_df.to_string(index=False, float_format='%.4f'))
print("\nRisk-Free Rate Sensitivity:")
print(rf_df.to_string(index=False, float_format='%.4f'))

In [ ]:
# Sensitivity charts
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(vol_df['Volatility']*100, vol_df['ESO Value'], 'o-', color='#1F3864', linewidth=2)
axes[0,0].set_title('ESO Value vs Volatility', fontweight='bold')
axes[0,0].set_xlabel('Volatility (%)')
axes[0,0].set_ylabel('ESO Value (€)')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(life_df['Expected Life'], life_df['ESO Value'], 's-', color='#2E75B6', linewidth=2)
axes[0,1].set_title('ESO Value vs Expected Life', fontweight='bold')
axes[0,1].set_xlabel('Expected Life (years)')
axes[0,1].set_ylabel('ESO Value (€)')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(vest_df['Vesting Period'], vest_df['ESO Value'], '^-', color='#C00000', linewidth=2)
axes[1,0].set_title('ESO Value vs Vesting Period', fontweight='bold')
axes[1,0].set_xlabel('Vesting Period (years)')
axes[1,0].set_ylabel('ESO Value (€)')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(rf_df['Risk-Free Rate']*100, rf_df['ESO Value'], 'D-', color='#2E8B57', linewidth=2)
axes[1,1].set_title('ESO Value vs Risk-Free Rate', fontweight='bold')
axes[1,1].set_xlabel('Risk-Free Rate (%)')
axes[1,1].set_ylabel('ESO Value (€)')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('Q6(iii): Executive Stock Option — Multi-Parameter Sensitivity', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()


**Comprehensive Interpretation.**

Hull-White (2004) ESO model calculates the Black-Scholes value less the specific behaviour of employees:

**Expected life:** contractual life Employees tend to exercise early (at about 2.5x strike), which lowers the time value of the option. A 6.5-year life of the expected life will result in a lesser value as compared to 10 years.

Forfeiture in the course of vesting: Forfeiture in 3 years at a rate of 3% decreases the value expected by approximately 9% (survival probability [?] 0.91).

**Volatility (vega effect):** The most important driver, which is an increase in volatility raises the value of the ESO, which provides an incentive to take risks that were reported by Guay (1999) and Coles et al. (2006). This is why regulators (SEC, FASB ASC 718) need fair-value expensing.

**Vesting period:** The longer the vesting, the greater the value lost through increased forfeiture likelihood and time-discounting, the more the incentive of the employee corresponds with long-term shareholder value.

**Regulatory Perspective:**
FASB ASC 718 (previously SFAS 123R): It obligates the expensing of grants at fair-value at the time of granting, and such expensing is required to be recorded in the income statement.
SEC disclosure: The companies should reveal the valuation assumptions (volatility, expected life, forfeiture rate).
EU 28th Regime (EU-Inc): The harmonisation of ESOP treatment suggested by the European Commission to the member states will cross-border friction of stock-based compensation, which may enhance talent acquisition of EU startups competing with US equity compensation plans.

**Q6(iii) interpretation**

This analysis values the executive stock option first as a common dividend-paying call option as per the Black-Scholes model and subsequently incorporates employee-specific characteristics. Substituting the expected exercise life of 6.5 years for the contractual life of 10 years in the Black-Scholes formula gave a value of about €17.5830. Reduction of the ESO fair value to about €16.04 by applying a survival factor for vesting based on a 3.5% annual forfeiture rate over a 3-year period was due to the fact that the ESO may be forfeited before vesting. This makes economic sense because the employee may not survive the vesting period and the ESO holder cannot buy or sell the ESO, or hedge the option, as a market participant might.

The price difference between intrinsic value (€2.00) and the fair value (€16.04) indicates that the option's value is largely due to time value. This illustrates the role of uncertainty, expected life and volatility in determining the value of ESOs. Much of the option value comes from the potential for the share price to move favourably in the future rather than its current intrinsic value.

The sensitivity analysis shows volatility is critical to ESO value. Higher volatility leads to a significant increase in ESO value because there is an increase in the possible upside subject to an upper bound on the holder's downside. It also confirms that greater expected life increases the value because it increases the likelihood that the share price will increase before the option is expected to be exercised. On the other hand, the value of an ESO decreases when the expected life increases because the shares are not available for exercise for longer and the risk of the option forfeiture increases over the restricted period.

In accounting and regulatory terms, the fair-value measurement is important because the ESO expense impacts earnings, executive remuneration and disclosure, and corporate governance. The value of options should thus account not only for the market factors of share price, volatility, dividends and interest rates, but also for the behavioural and contractual characteristics of vesting, forfeiture and early exercise. For this reason, the value of an ESO is generally not equal to the value of a readily tradable vanilla option.

While the modified Black-Scholes model is intuitive and can be readily programmed in Excel and Python, it is still an approximation. It assumes volatility is constant and does not endogenise early exercise. In reality, firms tend to use more sophisticated modelling techniques, such as binomial trees or simulations in order to better model exercise behaviour. However, the proxy method used in this paper can be used to estimate a fair value for reporting and sensitivity purposes.

## Cross-Verification Summary

| Model | Excel Value | Python Value | Match? |
|-------|------------|-------------|--------|
| Q5(i) Annuity PV | €41,658.67 | €41,658.67 | ✓ |
| Q5(ii) Bond Price | €977.56 | €977.56 | ✓ |
| Q5(ii) Macaulay Duration | 5.8360 yrs | 5.8360 yrs | ✓ |
| Q5(ii) Modified Duration | 5.6605 yrs | 5.6605 yrs | ✓ |
| Q5(iii) Converged YTM | 5.1742% | 5.1734% | ✓ |
| Q5(iv) Monthly Payment | €1,890.68 | €1,890.68 | ✓ |
| Q5(v) 10yr Spot Rate | 4.2251% | 4.2251% | ✓ |
| Q6(i) Black-76 Call | €6.7652 | €6.7652 | ✓ |
| Q6(i) Black-76 Put | €5.5941 | €5.5941 | ✓ |
| Q6(ii) 10-day VaR (99%) | €317,773.14 | €317,773.14 | ✓ |
| Q6(ii) 10-day CVaR (99%) | €367,854.88 | €367,854.88 | ✓ |
| Q6(iii) Black-Scholes Value | €15.2899 | €15.2899 | ✓ |
| Q6(iii) ESO Fair Value | €13.7659 | €13.7659 | ✓ |

All Python outputs are consistent with Excel formulas to at least four decimal places, confirming the accuracy and reproducibility of both implementations.